In [278]:
# the goal is simple take a bunch of people and pridict 0 or 1

In [279]:
# output format 
# PassengerId,Survived
#892,0
#893,1
#894,0|

# features that matters(sex sibsp, Embarked)

In [280]:
# load and preprocess the data

In [281]:
import numpy as np 
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [282]:
train_data = pd.read_csv("train.csv")

In [283]:
test_data = pd.read_csv("test.csv")

In [284]:
features = ["Pclass", "SibSp", "Parch", "Sex", "Embarked", "Age", "Fare"]

In [285]:
def preprocess(data): 
    data["Sex"] = data["Sex"].map({"male": 1, "female": 0})
    data["Embarked"] = data["Embarked"].fillna("S")
    data["Embarked"] = data["Embarked"].map({"C": 0, "Q": 1, "S": 2})
    data["Age"] = data["Age"].fillna(data["Age"].median())
    data["Fare"] = data["Fare"].fillna(data["Fare"].median())
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
    data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

    mapping = {
    'Mr': 0, 'Miss': 1, 'Mrs': 2,
    'Master': 3, 'Dr': 4, 'Rev': 4, 'Col': 4, 'Major': 4,
    'Mlle': 1, 'Ms': 1, 'Lady': 2,
    'Sir': 4, 'Countess': 2, 'Jonkheer': 4, 'Don': 4
    }
    data['Title'] = data['Title'].map(mapping).fillna(4).astype(int)
    data["FareBin"] = pd.qcut(data["Fare"], 4, labels=False)
    data["AgeBin"] = pd.cut(data["Age"], 5, labels=False)
    data["CabinLetter"] = data["Cabin"].str[0]
    data["CabinLetter"] = data["CabinLetter"].map({
    'A':1,'B':2,'C':3,'D':4,'E':5,'F':6,'G':7,'T':8
    }).fillna(0).astype(int)



    return data

In [286]:
train_proc = preprocess(train_data.copy())
test_proc  = preprocess(test_data.copy())

train_X = train_proc[features]
train_y = train_proc["Survived"]
test_X  = test_proc[features]

In [287]:
# here comes the model 

In [288]:
model = XGBClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.1,
    objective="binary:logistic"
)


In [289]:
model.fit(train_X, train_y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=0.1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=2, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=600,
              n_jobs=None, num_parallel_tree=None, ...)

In [290]:
preds = (model.predict_proba(test_X)[:, 1] > 0.5).astype(int)

In [291]:
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": preds
})

submission.to_csv("submission.csv", index=False)

print("submission.csv 已生成")



submission.csv 已生成
